# AML_05 — Boucles de rétroaction et budget de calcul

*Conception pédagogique : Mathieu Weill. Usage personnel d’apprentissage ; droits et crédits tiers conservés.*

**Compétition de travail : [fraud](https://www.kaggle.com/competitions/playground-series-s6e11)**

**Question : Quelle prochaine expérience a le plus de valeur ?**

7 heures pédagogiques, hors pauses. Les exemples sont des jalons techniques courts ; l’essentiel du temps est consacré aux exercices, revues et expériences. **Version étudiant.**

**[Repère de compréhension : Choisir les expériences et leur budget](../../../course/chapitres/AML_05/README.md)**


## Lecture accompagnée

Avant le code, lire avec le professeur le [chapitre 5 du cours rédigé](../../../course/chapitres/AML_05/cours.pdf), section par section. Répondre à une question, corriger ensemble, puis retrouver le même raisonnement dans le notebook. Le PDF complet est disponible dans le dossier `student/lectures`.

## Parcours guidé — Choisir une expérience et un budget

**Objectif minimal :** Proposer une hypothèse ciblée et un budget court. Interpréter un échec et une mesure de ressources, sans inventer la mémoire GPU.

**Idée à retenir :** Une amélioration doit répondre à une erreur observée et être comparée sur un protocole stable.

Nous avançons en cinq étapes : **je montre → nous faisons ensemble → vous reproduisez → vous changez une chose → vous expliquez**. Ne pas lancer les autres cellules avant le signal de l’enseignant. Le premier exemple est volontairement minuscule et distinct des données de compétition.

### Exemple au tableau

Reprendre les six transactions de J1. Au seuil 0,50 : une fausse alerte et une fraude ratée. Fixer des coûts pédagogiques : fausse alerte=2 ; fraude ratée=20. Coût total=22.

Au seuil 0,30 : alertes B,D,E ; deux fraudes détectées et une fausse alerte ; coût total=2. Le modèle et son AUC n’ont pas changé.

Écrire à côté : « coût hypothétique, distinct de la métrique officielle Kaggle ». Le meilleur seuil dépend de l’objectif et ne se choisit pas sur le test.

Budget : 12 configurations × 3 folds × 2 seeds = 72 fits. À 4 min par fit : 288 min, soit 4 h 48 séquentielles, hors autres coûts.

### Répondez avant de lancer Python

1. Qu’avons-nous modifié : le modèle ou la règle d’action ?
2. Pourquoi un score AUC ne suffit-il pas à choisir un seuil ?
3. Qu’écrit-on avant de lancer une nouvelle expérience ?
4. Quand faut-il abandonner une expérience ?

## Préparation et limites

Lire README.md, compétitions et règles. Les données officielles sont téléchargées personnellement et placées dans data/competitions/<slug>/. Aucun remplacement silencieux par des données simulées. Poids préentraînés à préparer avant NLP_02/CV_01/CV_02. Les limites de calcul et d’échanges dépendent du concours et du compte.

## Notions du jour

Une boucle utile part d’une erreur observée, formule une hypothèse, prévoit une mesure et un budget, puis décide. Douze assistants de code ne remplacent pas un protocole partagé.

Comparer par sous-groupes, mais annoncer les sous-groupes et leur effectif : chercher après coup un segment spectaculaire peut tromper. Une importance par permutation décrit la dépendance d’un modèle à une variable sous un protocole donné, pas un lien causal.

Un ensemble de probabilités doit aligner **observations et classes**. Choisir ses poids sur des prédictions OOF, pas sur le test. La précision mixte, le cache et les checkpoints ont des usages distincts ; le GPU n’accélère pas tous les modèles.

Ne pas additionner des quotas supposés : mesurer les accès et contraintes du compte et du concours, garder les comptes individuels et respecter les règles de partage.

In [ ]:
from pathlib import Path
import sys,os,json,time
import numpy as np,pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
HERE=Path.cwd()
ROOT=next((p if (p/'mllab/competition.py').exists() else p/'student') for p in [HERE,*HERE.parents] if (p/'mllab/competition.py').exists() or (p/'student/mllab/competition.py').exists())
sys.path.insert(0,str(ROOT))
from mllab.competition import SPECS,read_competition,protocol,run_competition,binary_scores,get_folder
KEY='fraud';SPEC=SPECS[KEY]
QUICK=os.environ.get('MLLAB_QUICK','0')=='1'
LIMIT=int(os.environ.get('MLLAB_LIMIT','1000' if QUICK else '15000'))
IMAGE_LIMIT=int(os.environ.get('MLLAB_IMAGE_LIMIT','24' if QUICK else '240'))
plt.rcParams.update({'figure.figsize':(10,4),'axes.spines.top':False,'axes.spines.right':False})
print('Compétition :',SPEC['slug'],'métrique :',SPEC['metric'],'sens :',SPEC['direction'])
if KEY!='catsdogs':a,test,sample,protocol_record=protocol(KEY)
else:
 from mllab.vision_competition import image_protocol
 image_protocol(limit=IMAGE_LIMIT)
print('Données officielles trouvées. Profil rapide :',QUICK,'; aucun score Kaggle inventé.')

### Retrouver le tableau en Python

Lire les entrées ; prévoir la sortie ; exécuter ; comparer avec le calcul manuel.

In [ ]:
# Montrer que décision et score de classement répondent à deux questions.
from sklearn.metrics import confusion_matrix,roc_auc_score
y_small=np.array([0,1,0,1,0,0]);p_small=np.array([.10,.70,.20,.40,.60,.05])
for threshold in [.5,.3,.8]:
 tn,fp,fn,tp=confusion_matrix(y_small,p_small>=threshold).ravel()
 print('Seuil',threshold,'FP',fp,'FN',fn,'coût hypothétique',2*fp+20*fn)
print('ROC-AUC inchangée :',roc_auc_score(y_small,p_small))

## Analyser les erreurs et la précision/rappel

**Point d’arrêt avant le calcul 1.**

1. L’enseignant annonce la question traitée et montre les entrées utilisées.
2. Vous dites ce qui est appris sur le train et ce qui est seulement mesuré.
3. Vous exécutez sans changer le code ; vous repérez une sortie et son unité.
4. Vous expliquez ce que cette sortie permet de conclure et ce qu’elle ne prouve pas.

**Ne pas enchaîner si vous ne pouvez pas nommer la sortie.** Le professeur fait un exemple analogue ; les plus avancés ont une extension facultative.

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay,ConfusionMatrixDisplay
out,scores,fitted=run_competition(KEY,model='boosting',limit=LIMIT,hypothesis='Identifier les erreurs avant de proposer une nouvelle expérience.')
pred=pd.read_csv(out/'predictions.csv')
PrecisionRecallDisplay.from_predictions(pred.target,pred.probability);plt.show()
ConfusionMatrixDisplay.from_predictions(pred.target,pred.probability>=.5);plt.show()
display(pred.assign(error=(pred.probability>=.5)!=pred.target).sort_values('probability',ascending=False).head(10))
print('La métrique officielle est ROC-AUC ; le choix d’un seuil relève d’un usage différent et ne change pas le classement AUC.')

## Ensemble exploratoire et coût

**Point d’arrêt avant le calcul 2.**

1. L’enseignant annonce la question traitée et montre les entrées utilisées.
2. Vous dites ce qui est appris sur le train et ce qui est seulement mesuré.
3. Vous exécutez sans changer le code ; vous repérez une sortie et son unité.
4. Vous expliquez ce que cette sortie permet de conclure et ce qu’elle ne prouve pas.

**Ne pas enchaîner si vous ne pouvez pas nommer la sortie.** Le professeur fait un exemple analogue ; les plus avancés ont une extension facultative.

In [ ]:
paths=[];rows=[]
for model in ['linear','boosting']:
 path,score,_=run_competition(KEY,model=model,limit=LIMIT,hypothesis='Chercher des erreurs complémentaires, sans toucher au test.')
 paths.append(path);rows.append({'model':model,**score,'seconds':json.loads((path/'metrics.json').read_text())['seconds']})
p1=pd.read_csv(paths[0]/'predictions.csv');p2=pd.read_csv(paths[1]/'predictions.csv')
assert p1.id.equals(p2.id) and p1.target.equals(p2.target)
print('Moyenne simple',binary_scores(p1.target.to_numpy(),(p1.probability.to_numpy()+p2.probability.to_numpy())/2))
display(pd.DataFrame(rows))
print('Validation exploratoire déjà consultée : choisir ensuite les poids via OOF et réserver un audit final. Gain non garanti.')

## Extensions facultatives — après validation du socle

Ces pistes servent aux étudiants déjà à l’aise. Elles ne sont pas à réaliser toutes ni nécessaires pour accéder à la séance suivante. Le travail obligatoire est celui indiqué dans le parcours guidé et la fiche de séance.

1. Rédiger deux hypothèses basées sur les erreurs et choisir celle à tester d’abord avec justification.
2. Construire un tableau score/temps/mémoire ; noter explicitement les mesures absentes au lieu de les inventer.
3. Définir trois critères d’arrêt d’une expérience et tester une reprise de checkpoint sur un dossier de démonstration.

Pour chaque essai : hypothèse avant calcul, code/configuration, empreinte du split, métrique officielle, durée, budget, conclusion, prochaine expérience. Chaque membre doit pouvoir expliquer son apport.

**Point d’arrêt avant le calcul 3.**

1. L’enseignant annonce la question traitée et montre les entrées utilisées.
2. Vous dites ce qui est appris sur le train et ce qui est seulement mesuré.
3. Vous exécutez sans changer le code ; vous repérez une sortie et son unité.
4. Vous expliquez ce que cette sortie permet de conclure et ce qu’elle ne prouve pas.

**Ne pas enchaîner si vous ne pouvez pas nommer la sortie.** Le professeur fait un exemple analogue ; les plus avancés ont une extension facultative.

In [ ]:
HYPOTHESE = 'À écrire avant le prochain calcul'
CONCLUSION = 'À compléter avec une preuve et une limite'
# Ajouter ici une expérience limitée, puis expliquer la décision de la conserver ou non.

## Questions de sortie

- Deux modèles qui font les mêmes erreurs apportent-ils beaucoup à un ensemble ?
- Comment sais-tu que le gain n’est pas simplement une autre seed ?

**Remise :** un run traçable, une conclusion, un artefact ou une soumission valide et une prochaine hypothèse. Aucun score public ni prix garanti.